# Explainable AI with SHAP

Understanding why the model ranked a customer highly.

In [ ]:
import pandas as pd
import numpy as np
import shap
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split

df = pd.read_csv(
    "../data/processed/model_data.csv"
)

X = df.drop(columns=["y"])
y = df["y"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# To extract preprocessing and model, we need the trained model from the previous notebook.
# Since we don't have it explicitly saved, let's retrain it quickly for this notebook context.

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier

categorical_features = X.select_dtypes(include=["object", "category"]).columns.tolist()
numerical_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numerical_features),
    ("categorical", categorical_pipeline, categorical_features)
])

xgb_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", XGBClassifier(n_estimators=400, max_depth=6, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8, eval_metric="logloss", random_state=42))
])

xgb_model.fit(X_train, y_train)

In [ ]:
preprocessor = xgb_model.named_steps["preprocessor"]
model = xgb_model.named_steps["model"]

X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

feature_names = (
    preprocessor
    .get_feature_names_out()
)

print(len(feature_names))

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_transformed)

In [ ]:
shap.summary_plot(
    shap_values,
    X_test_transformed,
    feature_names=feature_names
)

In [ ]:
importance = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0)
})

importance = importance.sort_values(
    "mean_abs_shap",
    ascending=False
)

importance.head(20)

In [ ]:
customer_index = 0

customer = X_test.iloc[customer_index]
customer

In [ ]:
customer_transformed = X_test_transformed[customer_index]
customer_shap = shap_values[customer_index]

customer_explanation = pd.DataFrame({
    "feature": feature_names,
    "shap_value": customer_shap
})

customer_explanation["abs_shap"] = customer_explanation["shap_value"].abs()

customer_explanation = (
    customer_explanation
    .sort_values("abs_shap", ascending=False)
)

customer_explanation.head(10)

In [ ]:
positive_factors = (
    customer_explanation[
        customer_explanation["shap_value"] > 0
    ]
    .sort_values("shap_value", ascending=False)
)

positive_factors.head(5)

In [ ]:
negative_factors = (
    customer_explanation[
        customer_explanation["shap_value"] < 0
    ]
    .sort_values("shap_value")
)

negative_factors.head(5)

In [ ]:
from src.models.explain_customer import explain_customer

explanation = explain_customer(customer_shap, feature_names)
explanation

In [ ]:
from src.decision_engine.campaign_optimizer import CampaignConfig, rank_customers

config = CampaignConfig(conversion_value=1000, contact_cost=20, budget=100000)

all_probability = xgb_model.predict_proba(X)[:, 1]
decision_df = X.copy()
decision_df["xgb_probability"] = all_probability

ranked_customers = rank_customers(
    decision_df,
    probability_column="xgb_probability",
    config=config
)

In [ ]:
def assign_priority(probability, expected_value):
    if expected_value <= 0:
        return "DO NOT CONTACT"
    if probability >= 0.70:
        return "HIGH"
    if probability >= 0.40:
        return "MEDIUM"
    return "LOW"

ranked_customers["priority"] = ranked_customers.apply(
    lambda row: assign_priority(row["xgb_probability"], row["expected_value"]),
    axis=1
)

ranked_customers[
    ["xgb_probability", "expected_value", "priority"]
].head(20)